# Simulação de compra: Financiar ou pagar à Vista?

## Livrarias:

In [1]:
import scr.loan as loan
import scr.equity as eq
import scr.vis as vis
import numpy as np

## Parâmetros fixos:

P0: O patrimônio inicial dessa simulação será sempre igual ao Bem comprado, pois essa simulação é somente para comparação entre os métodos, porém, se alguém quiser usar um outro valor de patrimônio inicial pode também, só os valores finais serão correspondentemente aumentados.

Am: Aporte mensal também pode ser escolhido pelo indivíduo. Um investidor inteligente separaria, ao menos, 10% de seu salário para aportes.

rm: Rendimento mensal esperado não depende da situação financeira do indivíduo, somente de sua política de investimentos.

mm: Aumento de aporte mensal esperado depende da situação financeira do indivíduo, pois evolução de carreira profissional é o método principal de aumento de aporte, porém disciplina e psicologia financeira têm o potencial de impor um valor fixo, independente das circunstâncias.

Tc: O indivíduo pode escolher o quanto tempo passará pesquisando ou guardando enquanto espera o momento certo.

Jm: Juros mensais depende do humor do agiota.

Th: Horizonte escolhido para rodar a simulação.

Vp: Escolhido a partir da condição financeira do indivíduo, porém, se for pequena tal que o empréstimo seja impossível, uma mensagem de erro aparecerá e dirá o mínimo aceitável.

### Observação: Esta simulação responderá se vale, ou não pagar à vista um por um Bem, por isso é suposto que P0 >= Valor à Vista do Bem.

## Exemplo:

In [2]:
Am_1 = 500.00
rm_1 = eq.anual_para_mensal(1.10)
mm_1 = eq.anual_para_mensal(1.05)
Th_1 = 12*6
Vv_1 = 200e3 
Tc_1 = 4
Jm_1 = 1.01
pm_1 = eq.anual_para_mensal(0.9)
Gr_1 = 3000
Vm_1 = Vv_1*0.01/12
P0_1 = Vv_1
Vp_1 = 2000

### Sem compra:

In [3]:
print(f'Patrimônio final no horizonte: R$ {eq.patrimonio_puro(P0_1, Am_1, rm_1, mm_1,Th_1):.2f}')

Patrimônio final no horizonte: R$ 410739.18


### Compra:

In [4]:
# Cálculo da diferença entre os métodos de pagamento:

def delta_financiamento_menos_vista(valor_vista,
                    aporte_mensal,
                    rendimento_mensal,
                    aumento_de_aporte_mensal,
                    mes_compra,
                    valor_parcela,
                    valor_manutencao,
                    ganho_problema,
                    horizonte,
                    juros_mensais,
                    preciacao,
                    print_info):
    
    entrada_minima = valor_vista * 0.10
    
    parcela_minima = loan.valor_parcela(valor_vista, entrada_minima, juros_mensais, 420)
    if valor_parcela < parcela_minima:
        valor_parcela = parcela_minima
        if print_info == True:
            print(f'Valor da parcela ajustado para o mínimo exigido: R$ {valor_parcela:.2f}')
    elif valor_parcela > parcela_minima:
        if print_info == True:
            num_parcelas = loan.encontrar_numero_parcelas(loan.tabela_comparativa(valor_vista, entrada_minima, juros_mensais), valor_parcela)
            print(f'Duração do financiamento = {num_parcelas} meses com parcelas de R$ {valor_parcela:.2f}')
    
    aporte_minimo = aporte_mensal + parcela_minima + valor_manutencao

    pv = eq.patrimonio_final_sem_endividamento(valor_vista,
                                               aporte_minimo,
                                               rendimento_mensal,
                                               aumento_de_aporte_mensal,
                                               mes_compra,
                                               valor_vista,
                                               ganho_problema,
                                               valor_manutencao,
                                               horizonte,
                                               preciacao)
    
    pf = eq.patrimonio_total_com_endividamento(valor_vista,
                                               aporte_minimo,
                                               rendimento_mensal,
                                               aumento_de_aporte_mensal,
                                               horizonte,
                                               valor_vista,
                                               entrada_minima,
                                               valor_parcela,
                                               valor_manutencao,
                                               ganho_problema,
                                               mes_compra,
                                               juros_mensais,
                                               preciacao)
    
    if print_info == True:
        print(f'Condições iniciais: Valor do Bem = R$ {valor_vista:.2f}; Orçamento para Investimentos e Compra do Bem = R$ {aporte_minimo:.2f}')
        print(f'Patrimônio final no horizonte sem financiamento: R$ {pv:.2f}')
        print(f'Patrimônio final no horizonte com financiamento: R$ {pf:.2f}')
        if pf > pv:
            print(f'Financiamento é melhor do que pagamento à vista em R$ {pf - pv:.2f}')
        elif pf < pv:
            print(f'Pagamento à vista é melhor do que financiamento em R$ {pv - pf:.2f}')
        else:
            print('Financiamento e pagamento à vista são equivalentes.')
    
    return pf - pv

In [5]:
delta_financiamento_menos_vista(Vv_1, Am_1, rm_1, mm_1, Tc_1, 100, Vm_1, Gr_1, Th_1, Jm_1, pm_1, True)

Valor da parcela ajustado para o mínimo exigido: R$ 1827.99
Condições iniciais: Valor do Bem = R$ 200000.00; Orçamento para Investimentos e Compra do Bem = R$ 2494.66
Patrimônio final no horizonte sem financiamento: R$ 702005.47
Patrimônio final no horizonte com financiamento: R$ 1045797.21
Financiamento é melhor do que pagamento à vista em R$ 343791.74


343791.7393854888

In [6]:
# encontrar curva em que delta = 0:

def curva_delta_zero(aporte_mensal,
                    rendimento_mensal,
                    aumento_de_aporte_mensal,
                    mes_compra,
                    valor_parcela,
                    valor_manutencao,
                    ganho_problema,
                    horizonte,
                    preciacao):
    
    curva = []
    for valor_vista in np.arange(1e3, 1000e3, 10e3):
        coordenadas = []
        coordenadas.append(valor_vista)

        for juros_mensais in np.arange(1.01, 1.05, 0.001):
            delta = delta_financiamento_menos_vista(valor_vista,
                                                    aporte_mensal,
                                                    rendimento_mensal,
                                                    aumento_de_aporte_mensal,
                                                    mes_compra,
                                                    valor_parcela,
                                                    valor_manutencao,
                                                    ganho_problema,
                                                    horizonte,
                                                    juros_mensais,
                                                    preciacao,
                                                    False)
            
            if delta == 0:
                coordenadas.append(juros_mensais)
                break

        curva.append(coordenadas)


    return curva

In [7]:
curva_delta_zero(Am_1, rm_1, mm_1, Tc_1, 1, Vm_1, Gr_1, Th_1, pm_1)

[[np.float64(1000.0)],
 [np.float64(11000.0)],
 [np.float64(21000.0)],
 [np.float64(31000.0)],
 [np.float64(41000.0)],
 [np.float64(51000.0)],
 [np.float64(61000.0)],
 [np.float64(71000.0)],
 [np.float64(81000.0)],
 [np.float64(91000.0)],
 [np.float64(101000.0)],
 [np.float64(111000.0)],
 [np.float64(121000.0)],
 [np.float64(131000.0)],
 [np.float64(141000.0)],
 [np.float64(151000.0)],
 [np.float64(161000.0)],
 [np.float64(171000.0)],
 [np.float64(181000.0)],
 [np.float64(191000.0)],
 [np.float64(201000.0)],
 [np.float64(211000.0)],
 [np.float64(221000.0)],
 [np.float64(231000.0)],
 [np.float64(241000.0)],
 [np.float64(251000.0)],
 [np.float64(261000.0)],
 [np.float64(271000.0)],
 [np.float64(281000.0)],
 [np.float64(291000.0)],
 [np.float64(301000.0)],
 [np.float64(311000.0)],
 [np.float64(321000.0)],
 [np.float64(331000.0)],
 [np.float64(341000.0)],
 [np.float64(351000.0)],
 [np.float64(361000.0)],
 [np.float64(371000.0)],
 [np.float64(381000.0)],
 [np.float64(391000.0)],
 [np.float6

In [8]:
import numpy as np

def curva_delta_zero1(aporte_mensal,
                    rendimento_mensal,
                    aumento_de_aporte_mensal,
                    mes_compra,
                    valor_parcela,
                    valor_manutencao,
                    ganho_problema,
                    horizonte,
                    preciacao):
    
    curva = []

    for valor_vista in np.arange(1e3, 1000e3, 10e3):

        juros_range = np.arange(1.01, 1.05, 0.001)

        delta_anterior = None
        raiz_encontrada = None

        for juros_mensais in juros_range:

            delta = delta_financiamento_menos_vista(
                valor_vista,
                aporte_mensal,
                rendimento_mensal,
                aumento_de_aporte_mensal,
                mes_compra,
                valor_parcela,
                valor_manutencao,
                ganho_problema,
                horizonte,
                juros_mensais,
                preciacao,
                False
            )

            if np.isnan(delta) or np.isinf(delta):
                continue

            if delta_anterior is not None:

                # 🔥 detecta mudança de sinal
                if delta_anterior * delta < 0:

                    # interpolação linear simples
                    raiz = juros_mensais - 0.001 * delta / (delta - delta_anterior)

                    raiz_encontrada = raiz
                    break

            delta_anterior = delta

        if raiz_encontrada is not None:
            curva.append((valor_vista, raiz_encontrada))

    return np.array(curva)

In [9]:
curva_delta_zero1(Am_1, rm_1, mm_1, Tc_1, 1, Vm_1, Gr_1, Th_1, pm_1)

array([], dtype=float64)

In [10]:
def bissecao(f, a, b, tol=1e-6, max_iter=50):

    fa = f(a)
    fb = f(b)

    if np.isnan(fa) or np.isnan(fb):
        return None

    if fa * fb > 0:
        return None  # sem mudança de sinal

    for _ in range(max_iter):

        m = 0.5 * (a + b)
        fm = f(m)

        if np.isnan(fm):
            return None

        if abs(fm) < tol:
            return m

        if fa * fm < 0:
            b = m
            fb = fm
        else:
            a = m
            fa = fm

    return 0.5 * (a + b)

In [11]:
import numpy as np

def curva_delta_zero2(aporte_mensal,
                    rendimento_mensal,
                    aumento_de_aporte_mensal,
                    mes_compra,
                    valor_parcela,
                    valor_manutencao,
                    ganho_problema,
                    horizonte,
                    preciacao):

    curva = []

    valores_vista = np.arange(1e3, 1000e3, 10e3)
    juros_grid = np.arange(1.01, 1.05, 0.002)  # grid mais grosso

    for valor_vista in valores_vista:

        def f(j):
            return delta_financiamento_menos_vista(
                valor_vista,
                aporte_mensal,
                rendimento_mensal,
                aumento_de_aporte_mensal,
                mes_compra,
                valor_parcela,
                valor_manutencao,
                ganho_problema,
                horizonte,
                j,
                preciacao,
                False
            )

        raiz_encontrada = None

        # 🔍 etapa 1: varrer grid
        for j1, j2 in zip(juros_grid[:-1], juros_grid[1:]):

            d1 = f(j1)
            d2 = f(j2)

            if np.isnan(d1) or np.isnan(d2):
                continue

            # 🔥 mudança de sinal detectada
            if d1 * d2 < 0:

                # 🔥 etapa 2: refinar com bisseção
                raiz = bissecao(f, j1, j2)

                raiz_encontrada = raiz
                break

        if raiz_encontrada is not None:
            curva.append((valor_vista, raiz_encontrada))

    return np.array(curva)

In [12]:
import matplotlib.pyplot as plt

def plot_curva(curva):

    if len(curva) == 0:
        print("⚠️ Nenhuma curva encontrada")
        return

    V = curva[:,0]
    J = curva[:,1]

    plt.figure(figsize=(10,6))
    plt.plot(V, J, marker='')

    plt.xlabel("Valor do imóvel")
    plt.ylabel("Juros mensais (Δ=0)")
    plt.title("Curva de indiferença (grid + bisseção)")
    plt.grid()
    plt.show()

In [13]:
plot_curva(curva_delta_zero2(Am_1, rm_1, mm_1, Tc_1, 1, Vm_1, Gr_1, Th_1, pm_1))

⚠️ Nenhuma curva encontrada
